
# Clasificación de noticias con Transformers

Autor: Álvaro José Cabrera Lozano, Claudia Lorena Áragon, Josué Cobaleda
Curso: Procesamiento de Lenguaje Natural  
Universidad Icesi

Este notebook implementa un modelo Transformer entrenado desde cero para clasificación de noticias y compara su desempeño con modelos preentrenados como BERT.
# Transformers desde (casi) cero

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion3/1-transformers-from-scratch.ipynb)




#### Referencias
- Dataset: https://huggingface.co/datasets/Davlan/sib200
- [Attention is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)
- [The Illustrated Transformer – Jay Alammar](http://jalammar.github.io/illustrated-transformer/)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.oreilly.com/library/view/natural-language-processing/9781098103231/)
- [Tutorial 6: Transformers and Multi-Head Attention (UvA DL Tutorials)](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial6/Transformers_and_MHAttention.html)

In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages


In [ ]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]'==4.41.2 sentence-transformers

### Cargando el dataset
Este es un dataset pequeño de articulos de noticias en idioma español con sus respectivas categorías. El dataset está disponible en el HuggingFace Hub y puede ser fácilmente descargado con la librería.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Cargar dataset alternativo
dataset = load_dataset("Davlan/sib200", "spa_Latn", split="train")

# Renombrar columna para que coincida con el script del profesor
dataset = dataset.rename_column("category", "label")

dataset

Observemos uno de sus registros...

In [ ]:
dataset[0]

Para los efectos de esta tarea, nos servirán el texto y la categoría naturalmente.

A manera general, observemos que tan largos o cortos tienden a ser los textos.

In [ ]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

Estos valores son la cantidad de *caractéres* que tiene las secuencias. Una decisión ingenua pero útil en este momento podría ser ajustar la longitud de las secuencias que vamos a usar para el entrenamiento a unos 2000 tokens. Esto podría ser suficiente para capturar una porción significativa de los textos.

## Definiendo el Tokenizer

Ahora, vamos a definir el tokenizer para nuestra tarea. Para ahorrarnos tiempo, vamos a entrenar uno basado en gpt2, pero ajustandolo a nuestro dataset. Para ello, debemos seleccionar una muestra representativa de nuestro dataset, como no es muy grande, casi que podemos usarlo todo. Luego, debemos definir el tamaño del vocabulario, es decir, cuantos tokens únicos queremos soportar en nuestro tokenizador. Para que un modelo de lenguaje funcione moderadamente bien para una tarea de clasificación, considerando el tamaño de nuestro corpus, deberíamos definir unos 50 mil tokens.

In [ ]:
import matplotlib.pyplot as plt

plt.hist(text_lengths, bins=30)
plt.title("Distribución de longitud de textos")
plt.xlabel("Número de caracteres")
plt.ylabel("Frecuencia")
plt.show()

### Interpretación de la distribución de longitud de textos

El histograma muestra la distribución del número de caracteres en los textos del dataset.
Se observa que la mayoría de los documentos se concentra aproximadamente entre **100 y 200 caracteres**, con un pico alrededor de los **130–160 caracteres**, lo que coincide con la longitud promedio calculada previamente.

Esto indica que el dataset está compuesto principalmente por textos relativamente cortos, lo cual es consistente con titulares o fragmentos breves de noticias.
Asimismo, se observa una ligera cola hacia la derecha, lo que sugiere la presencia de algunos textos más largos, aunque estos son menos frecuentes.

Esta característica es favorable para modelos basados en **Transformers**, ya que permite procesar la mayoría de los textos sin necesidad de utilizar secuencias muy largas durante la tokenización, reduciendo el uso de padding y mejorando la eficiencia del entrenamiento.

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from transformers.models.gpt2.tokenization_gpt2 import bytes_to_unicode

length = len(dataset)

tokenizer = AutoTokenizer.from_pretrained("gpt2")

byte_to_unicode_map = bytes_to_unicode()
unicode_to_byte_map = dict((v, k) for k, v in byte_to_unicode_map.items())
base_vocab = list(unicode_to_byte_map.keys())

def batch_iterator(batch_size=10):
    for i in tqdm(range(0, len(dataset), batch_size)):
        yield dataset[i:i+batch_size]["text"]

spanish_news_tokenizer = tokenizer.train_new_from_iterator(
    batch_iterator(),
    vocab_size=50000,
    initial_alphabet=base_vocab
)

### Entrenamiento del tokenizer

En esta etapa se entrena un nuevo tokenizer utilizando como base el
tokenizador de GPT-2. El objetivo es adaptar el vocabulario del modelo
al corpus específico utilizado en este proyecto, compuesto por noticias
en español.

Para ello se utiliza la función `train_new_from_iterator`, que permite
construir un nuevo vocabulario a partir de los textos del dataset.
Los textos se procesan en batches para mejorar la eficiencia del
entrenamiento.

El tokenizer resultante captura subpalabras frecuentes del corpus y
genera un vocabulario de hasta **50 000 tokens**, lo que facilita que el
modelo Transformer represente adecuadamente los patrones lingüísticos
presentes en las noticias.

Exploremos ahora el tokenizador obtenido.

In [ ]:
tokens = sorted(spanish_news_tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)
print(f"Vocabulario: {spanish_news_tokenizer.vocab_size} tokens")
print("Primeros 15 tokens:")
print([f"{spanish_news_tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[:15]])
print("15 tokens de en medio:")
print([f"{spanish_news_tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[1000:1015]])
print("Últimos 15 tokens:")
print([f"{spanish_news_tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[-15:]])

### Exploración del vocabulario del tokenizer

El tokenizer entrenado generó un vocabulario de **10 729 tokens**, lo cual indica
que el modelo aprendió un conjunto de subpalabras y caracteres frecuentes en el
corpus de noticias en español utilizado.

Al observar los **primeros tokens**, se identifican principalmente **caracteres
especiales y signos de puntuación**, como `!`, `#`, `$`, `&`, `(` y `)`. Esto es
esperado, ya que los tokenizadores basados en BPE o similares suelen incluir
estos símbolos básicos en el vocabulario inicial.

En los **tokens de la parte media del vocabulario** aparecen palabras comunes del
idioma español como *tener*, *debe*, *cerca*, *agua*, *científicos* y *gobier*,
lo cual sugiere que el tokenizer logró capturar términos frecuentes presentes
en el dataset.

Finalmente, en los **últimos tokens del vocabulario** se observan palabras menos
frecuentes o más específicas, como nombres propios o términos poco comunes
(por ejemplo *Kriegsmarine*, *MetroPlus*, *mosasáuridos*). Esto es característico
de los algoritmos de tokenización que construyen el vocabulario a partir de la
frecuencia de aparición de las subpalabras en el corpus.

En conjunto, estos resultados muestran que el tokenizer ha aprendido una
representación adecuada del vocabulario del dataset de noticias en español.

In [ ]:
spanish_news_tokenizer.pad_token = '[PAD]'
spanish_news_tokenizer("hola mundo!", max_length=8, truncation=True, padding='max_length')

Lo que obtenemos de vuelta son los ids de cada token según el vocabulario. Ahora algo importante que notamos aquí es el *padding*, durante el entrenamiento, queremos que las secuencias sean de tamaño fijo, para asi operar comodamente con matrices. Pero ya vimos que no todos los textos tienen la misma longitud. Entonces que hacer? para los que son más largos que una longitud dada simplemente cortamos, pero para los que son más cortos, debemos *rellenar* lo faltante con un *token especial de relleno o padding*. Y es justo lo que definimos allí, cuando la cadena es inferior a 8 **tokens**, entonces debemos hacer padding hasta que se cumplan los 8.

Ahora, notemos que "hola mundo!" son 2 palabras, 9 letras, 1 espacio y 1 simbolo para un total de 11 caracteres, pero vemos que el resultado son 4 tokens y el padding. Esto es trabajo del tokenizador. Cuando lo entrenamos con nuestro corpus, el tokenizador computó las frecuencias de palabras y sus partes, tal como vimos arriba, entonces, estos tokens juntos forman la frase original, observemos:

### Explicación del `attention_mask`

En los modelos basados en **Transformers**, las secuencias de texto deben tener
una longitud uniforme para poder ser procesadas en paralelo. Para lograr esto,
se utiliza **padding**, que consiste en añadir tokens especiales (`[PAD]`)
cuando el texto es más corto que la longitud máxima definida.

Sin embargo, estos tokens de padding no contienen información real del texto.
Por esta razón se utiliza el **attention mask**, que indica al modelo qué
posiciones de la secuencia deben ser consideradas durante el proceso de
atención.

El `attention_mask` es un vector binario donde:

- **1** indica que el token corresponde a una palabra real del texto.
- **0** indica que el token es padding y debe ser ignorado por el modelo.

Por ejemplo, en el resultado obtenido:


In [ ]:
spanish_news_tokenizer("hola mundo!", max_length=8, truncation=True, padding='max_length').tokens()

Claramente vemos los 4 tokens como cadenas independientes.

### Definiendo el dataset de pytorch
Ahora podemos proceder a definir el dataset. Esto debería ser muy sencillo dado que nuestro dataset es pequeño y ya tenemos el tokenizador listo.

In [ ]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset


class SpanishNewsDataset(Dataset):

    def __init__(self, tokenizer, dataset, seq_length: int = 512):
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = '[PAD]'
        self.dataset = dataset
        self.seq_length = seq_length

        # Mapas para convertir las etiquetas a números
        self.id_2_class_map = dict(enumerate(np.unique(dataset[:]['label'])))
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        self.num_classes = len(self.id_2_class_map)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text, y = self.dataset[index]['text'], self.dataset[index]['label']
        y = self.class_2_id_map[y]

        data = {
            k: torch.tensor(v)
            for k, v in self.tokenizer(
                text,
                max_length=self.seq_length,
                truncation=True,
                padding="max_length"
            ).items()
        }

        data['y'] = torch.tensor(y)

        return data

    def __len__(self):
        return len(self.dataset)

In [ ]:
import torch
import numpy as np
from typing import Tuple, Dict
from torch.utils.data import Dataset

class SpanishNewsDataset(Dataset):

    def __init__(self, tokenizer, dataset, seq_length: int = 512):
        self.tokenizer = tokenizer
        self.tokenizer.pad_token = '[PAD]'
        self.dataset = dataset
        self.seq_length = seq_length
        # Definimos estos dos mapas para facilitarnos la tarea
        # de traducir de nombres de categoría a ids de categoría.
        self.id_2_class_map = dict(enumerate(np.unique(dataset[:]['label'])))
        self.class_2_id_map = {v: k for k, v in self.id_2_class_map.items()}
        self.num_classes = len(self.id_2_class_map)

    def __getitem__(self, index) -> Dict[str, torch.Tensor]:
        text, y = self.dataset[index]['text'], self.dataset[index]['label']
        y = self.class_2_id_map[y]
        data = {k: torch.tensor(v) for k, v in self.tokenizer(text, max_length=self.seq_length, truncation=True, padding='max_length').items()}
        data['y'] = torch.tensor(y)
        return data


    def __len__(self):
        return len(self.dataset)

Ahora instanciaremos el dataset entero. Para este experimento, definiremos un tamaño máximo de secuencia de 2048 **tokens**. Que según nuestra intuición arriba, debería ser suficiente para la tarea.

In [ ]:
max_len = 2048
spanish_news_dataset = SpanishNewsDataset(spanish_news_tokenizer, dataset, seq_length=max_len)
assert len(spanish_news_dataset) == len(dataset)

Y luego, procedemos a hacer el train-val-test split y crear los dataloaders.

In [ ]:
from torch.utils.data import random_split
from torch.utils.data import DataLoader

batch_size = 4 if not IN_COLAB else 16
train_dataset, val_dataset, test_dataset = random_split(spanish_news_dataset, lengths=[0.8, 0.1, 0.1])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

### Definición de los Positional Embeddings

Según el paper, los autores agregan una secuencia sinusoidal a los embeddings de los tokens con el fin de inyectar información referente a la posición de cada token en las frases. Esto obedece a la definición:

$$
PE(pos, 2i) = \sin(pos/10000^{2i/d_{model}}) \\
PE(pos, 2i + 1) = \cos(pos/10000^{2i/d_{model}})
$$

Donde:
- $pos$ es la posición del *token* en la secuencia.
- $i$ es la dimensión $i$ en el embedding $d$.
- $d_model$ es la dimensionalidad total del embedding.

Lo que los autores propusieron fue que para las posiciones pares, se calculara el seno de la posición, relativa a la dimensionalidad del embedding y para las posiciones impares, se calculara el coseno. Según los autores, estos tenían la hipótesis de que estas funciones inyectarían la información posicional relativa de forma eficiente, en parte porque se pueden pre-calcular e inyectar directamente durante el entrenamiento, evitando asi emplear recursos en entrenar estructuras para aprenderlos.

Esto último es particularmente importante ya que se evita tanto hacer uso de recursos innecesarios como acelerar el proceso de entrenamiento al no tener que computar gradientes para esta parte. Sin embargo, los autores también mencionaron que es ciertamente posible aprender estos positional embeddings como parte del entrenamiento y que según sus resultados, no había mucha diferencia entre ambos enfoques, razón por la cual, se prefiere el positional encoding sinusoidal.

In [ ]:
import numpy as np
import torch.nn as nn
from enum import Enum
from typing import Optional


class PosEncodingType(Enum):
    SINUSOID = 1
    LEARNABLE = 2


class SinusoidPE(nn.Module):

    def __init__(self, max_len: int, d_model: int):
        super(SinusoidPE, self).__init__()

        # Definimos un vector columna con las posiciones de la secuencia de entrada (pos)
        pos = torch.arange(max_len).unsqueeze(1)
        # Definimos un vector de fila con las dimensiones del embedding (i)
        i = torch.arange(d_model).unsqueeze(0)

        # Calculamos el denominador segun la formula
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        # Aplicamos el denominador a las posiciones
        angle_rads = pos * div_term

        # Inicializamos la matriz de positional encodings
        pos_encoding = torch.zeros(max_len, d_model)
        # Calculamos los embeddings para los numeros pares con seno: PE(pos, 2i)
        pos_encoding[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        # Calculamos los embdeddings para los numeros inpares con coseno: PE(pos, 2i+1)
        pos_encoding[:, 1::2] = torch.cos(angle_rads[:, 1::2])

        # Registramos la variable como atributo de clase
        self.register_buffer("pos_encoding", pos_encoding.unsqueeze(0), persistent=False)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_encoding[:, :x.size(1), :]


class LearnablePE(nn.Module):

    def __init__(self, vocab_size: int, d_model: int, max_len: int = float('-inf')):
        super(LearnablePE, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(0, max(x.size(-1), self.max_len))
        pos_emb = self.embedding(positions)
        return x + pos_emb



class TokenAndPosEmbedding(nn.Module):

    def __init__(self, max_len: int, embed_dim: int, vocab_size: int, pos_encoding_type: PosEncodingType = PosEncodingType.SINUSOID):
        super(TokenAndPosEmbedding, self).__init__()
        self.token_emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        if pos_encoding_type == PosEncodingType.SINUSOID:
            self.pos_emb = SinusoidPE(max_len, embed_dim)
        else:
            self.pos_emb = LearnablePE(vocab_size, embed_dim)


    def forward(self, x):
        token_emb = self.token_emb(x)
        return self.pos_emb(token_emb)



Ahora procedemos a instanciar el modulo que va a convertir los tokens en embeddings con positional embeddings.

In [ ]:
emb_dim = 128 if not IN_COLAB else 256
tpe = TokenAndPosEmbedding(max_len, emb_dim, spanish_news_tokenizer.vocab_size)
pos_encoding = tpe.pos_emb.pos_encoding.squeeze(0).numpy()


A manera exploratoria, podemos observar gráficamente en que consisten estos vectores. En el siguiente gráfico podemos observar como los valores tienden a oscilar para diferentes posiciones en la dimensionalidad del embedding. Los valores individuales no tienen una interpretación directa, pero lo que vale la pena resaltar es que se observa una "transición" a medida que nos desplazamos por las dimensiones del embedding y sus respectivas posiciones, no es solo ruido.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
plt.pcolormesh(pos_encoding, cmap='viridis')
plt.xlabel('Embedding Dimensions')
plt.xlim((0, emb_dim))
plt.ylabel('Position')
plt.colorbar()
plt.show()

La figura muestra un mapa de calor del positional encoding sinusoidal utilizado por el modelo Transformer.

En el eje horizontal se representan las dimensiones del embedding, mientras que en el eje vertical se muestran las posiciones de los tokens dentro de la secuencia.

Los colores representan los valores generados por las funciones seno y coseno utilizadas para codificar la posición. Este patrón periódico permite que el modelo capture relaciones entre posiciones cercanas y lejanas dentro de la secuencia.

Ahora, si pasamos nuestra frase simple por el tokenizador, deberíamos obtener una matriz con la forma: $(longitud, d_{model})$:

In [ ]:
text = "hola mundo!"
tokens = spanish_news_tokenizer(text, max_length=max_len, truncation=True, padding='max_length')
x = torch.tensor(tokens['input_ids']).unsqueeze(0)
mask = torch.tensor(tokens['input_ids']).unsqueeze(0)
embedding = tpe(x)
embedding.shape

### Multi-Head Attention

![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/mh-attention.png?raw=1)

Ahora procedemos a definir al núcleo del modelo. Recodemos que la atención se define por:

$$
\text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_K}})V
$$

Que es la definición de "Scaled Dot-Product Attention". Y Multi-Head Attention es la concatenación de varias cabezas ejecutando el mismo scaled dot-product sobre partes del input. Entonces tenemos:

In [ ]:
import math


class MultiHeadAttention(nn.Module):

    def __init__(self, embed_size: int, num_heads: int = 8):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        assert embed_size & num_heads == 0, 'El tamaño del embedding debería ser divisible por el numero de cabezas'
        self.projection_dim = embed_size // num_heads
        self.query = nn.Linear(emb_dim, emb_dim)
        self.key = nn.Linear(emb_dim, emb_dim)
        self.value = nn.Linear(emb_dim, emb_dim)
        self.comibe_heads = nn.Linear(emb_dim, emb_dim)


    @staticmethod
    def _scaled_dot_product(q, k, v, mask=None):
        """scaled dot product.

        Esta función define el bloque mencionado.
        Aquí se hace la multiplicación de matrices
        entre los Q, K y V para luego calcular el
        score de atención.

        Nótese además que aquí aplicamos una máscara
        de atención. Esto se debe a que como estamos
        rellenando las cadenas cortas con un token que
        en si mismo no trae ningún significado, no queremos
        que la red desperdicie recursos operando sobre este
        token, entonces usamos la máscara para poner los valores
        de atención en numeros muy pequeños para que al
        calcular el score, estos no sobresalgan sobre los demás.
        """
        # d_k para el escalamiento
        d_k = q.size()[-1]

        # multiplicacion Q \cdot K^T
        attn_logits = torch.matmul(q, k.transpose(-2, -1))
        # escalamiento
        attn_logits = attn_logits / math.sqrt(d_k)

        # Se aplica la máscara
        if mask is not None:
            attn_logits = attn_logits.masked_fill(mask.reshape(mask.shape[0], 1, 1, -1) == 0, -9e-15)

        # Se calcula el score de atención.
        attention = torch.softmax(attn_logits, dim=-1)
        # Se obtienen los valores tras el score de atención.
        values = torch.matmul(attention, v)
        return values, attention


    def _separate_heads(self, x, batch_size):
        # Llega: (batch, seq_len, emb_dim)
        x =  x.reshape(batch_size, -1, self.num_heads, self.projection_dim)  # (batch, seq_len, num_heads, emb_dim / num_heads)
        return x.permute(0, 2, 1, 3)  # (batch, num_heads, seq_len, emb_dim / num_heads)


    def forward(self, x, mask=None, return_attention=False):
        """forward

        Este es todo el forward pass del multi-head attention.
        Aquí se coordina el resto de las operaciones, como
        la concatenación de las múltiples cabezas como
        el paso por la capa densa previo a entregar el
        resultado.
        """
        # x: (batch, seq_len, emb_dim)
        batch_size, seq_len, emb_dim = x.size()
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = self._separate_heads(q, batch_size)
        k = self._separate_heads(k, batch_size)
        v = self._separate_heads(v, batch_size)

        weights, attention = self._scaled_dot_product(q, k, v, mask)
        weights = weights.permute(0, 2, 1, 3) # (batch, seq_len, num_heads, emb_dim / num_heads)
        weights = weights.reshape(batch_size, seq_len, emb_dim)
        output = self.comibe_heads(weights)

        if return_attention:
            return output, attention
        else:
            return output


Podemos hacer una prueba rápida de que las operaciones funcionan a nivel de matrices.

In [ ]:
mha = MultiHeadAttention(emb_dim)
mha(embedding, mask).shape

### Definición del bloque transformers

![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/transformers-achitecture.png?raw=1)

Finalmente, definimos el bloque de transformers. Recordemos que como esta es una tarea de clasificación, solamente necesitamos el encoder, por lo que esto es silamente la primera parte del diseño de arquitecura de red.

En esta capa, simplemente ponemos una capa densa adicional junto con las normalizaciones a nivel de capa.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, emb_dim: int, num_heads: int = 8):
        super(TransformerBlock, self).__init__()
        self.mhatt = MultiHeadAttention(emb_dim, num_heads)
        self.mhatt_dropput = nn.Dropout(0.2)
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, emb_dim)
        )
        self.layer_norm1 = nn.LayerNorm(emb_dim)
        self.layer_norm2 = nn.LayerNorm(emb_dim)


    def forward(self, x, mask=None):
        attn_output = self.mhatt(x, mask)
        attn_output = self.mhatt_dropput(attn_output)
        attn_output = self.layer_norm1(attn_output)
        ffn_out = self.ffn(attn_output)
        return self.layer_norm2(ffn_out)


Nuevamente, probamos rapidamente para asegurarnos que las capas operan correctamente.

In [ ]:
tb = TransformerBlock(emb_dim)
tb(embedding, mask).shape

In [ ]:
num_heads = 8
vocab_size = spanish_news_tokenizer.vocab_size

token_embeddings = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
transformer = TransformerBlock(emb_dim, num_heads)
ff = nn.Sequential(
    nn.Flatten(),
    nn.Linear(max_len * emb_dim, spanish_news_dataset.num_classes)
)

In [ ]:
from torch.utils.data import DataLoader

batch_size = 16

train_loader = DataLoader(
    spanish_news_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    spanish_news_dataset,
    batch_size=batch_size
)

In [ ]:
it = iter(train_loader)
batch = next(it)
x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']

embeddings = token_embeddings(x)
assert embeddings.shape == (train_loader.batch_size, max_len, emb_dim)

attention = transformer(embeddings, mask)
attention.shape

| Variable | Qué contiene                       |
| -------- | ---------------------------------- |
| `x`      | tokens del texto                   |
| `mask`   | máscara de atención                |
| `y`      | etiqueta (categoría de la noticia) |


In [ ]:
pred = ff(attention)
pred.shape

En este paso se verifica el flujo completo del modelo. Primero se extrae un batch de datos del DataLoader, el cual contiene los input_ids, la attention_mask y las etiquetas de clase.

Los tokens se transforman en embeddings con dimensión (batch_size, max_len, emb_dim), obteniendo un tensor de tamaño (16, 2048, 256). Posteriormente estos embeddings se procesan mediante el Transformer, el cual mantiene la misma dimensionalidad.

Finalmente, la salida del Transformer se pasa por una capa feed-forward que produce las predicciones del modelo, generando un tensor de tamaño (16, 71), donde 71 corresponde al número de clases del problema de clasificación.

### Definición del clasificador

Finalmente, definimos el modelo en si. Este modelo constará de 3 capas:

- La tokenización, tal como la definimos anteriormente.
- El transformer, que acabamos de decinir.
- Una capa densa adicional que servirá como clasificador de aquello que nos entregue la capa del transformer.

Como este es un LightningModule, aquí definiremos el resto de funciones utilitarias para el entrenamiento de la tarea.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy


class SpanishNewsClassifier(LightningModule):

    def __init__(self, max_len: int, vocab_size: int, num_classes: int, emb_dim: int, num_heads: int = 8):
        super(SpanishNewsClassifier, self).__init__()
        self.num_classes = num_classes

        self.token_embeddings = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
        self.transformer = TransformerBlock(emb_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_len * emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
            nn.LogSoftmax(dim=1)
        )

        self.train_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc = Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc = Accuracy(task='multiclass', num_classes=num_classes)


    def forward(self, x, mask=None):
        embeddings = self.token_embeddings(x)
        attention = self.transformer(embeddings, mask)
        return self.classifier(attention)


    def training_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y)
        self.train_acc(y_hat, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y)
        self.val_acc(y_hat, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        self.test_acc(y_hat, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)


    def predict_step(self, batch):
        x, mask = batch['input_ids'], batch['attention_mask']
        return self(x, mask)


    def configure_optimizers(self):
        optimizer =  torch.optim.AdamW(self.parameters(), lr=2e-5, weight_decay=1e-5)
        return optimizer


model = SpanishNewsClassifier(max_len=spanish_news_dataset.seq_length, vocab_size=spanish_news_tokenizer.vocab_size, num_classes=spanish_news_dataset.num_classes, emb_dim=emb_dim)

tb_logger = TensorBoardLogger('tb_logs', name='TransformersClassifier')
callbacks=[EarlyStopping(monitor='train-loss', patience=3, mode='min')]
trainer = Trainer(max_epochs=10, devices=1, logger=tb_logger, callbacks=callbacks, precision="16-mixed")

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

El modelo Transformer fue entrenado durante 10 épocas utilizando GPU y precisión mixta. La arquitectura incluye una capa de embeddings con codificación posicional, un bloque Transformer para capturar relaciones contextuales y un clasificador feed-forward. El modelo contiene aproximadamente 271 millones de parámetros. Al finalizar el entrenamiento se obtuvo una precisión de aproximadamente 25% en entrenamiento y validación, lo que indica que el modelo logra capturar parcialmente los patrones del dataset, aunque aún existe margen para mejorar su desempeño.

Observemos el proceso de entrenamiento

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir tb_logs/

Y como es de esperarse, realizaremos la validación contra el conjunto de prueba.

In [ ]:
trainer.test(model, test_loader)

In [ ]:
from torch.utils.data import DataLoader

test_loader = DataLoader(
    spanish_news_dataset,
    batch_size=16
)

In [ ]:
model.eval()
trainer.test(model, test_loader)

### Haciendo predicciones

Finalmente, vamos a hacer uso del modelo y ver que tan bueno es para la clasificación de noticias.

In [ ]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [spanish_news_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

In [ ]:
from torch.utils.data import random_split

dataset_size = len(spanish_news_dataset)

train_size = int(0.8 * dataset_size)
val_size = int(0.1 * dataset_size)
test_size = dataset_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    spanish_news_dataset,
    [train_size, val_size, test_size]
)

In [ ]:
import pandas as pd

test_indices = test_dataset.indices

df = pd.DataFrame(data={
    "texto": dataset[test_indices]['text'],
    "tokens": [spanish_news_tokenizer(v)['input_ids'] for v in dataset[test_indices]['text']],
    "categoría": dataset[test_indices]['label'],
    "predicción": predictions[:len(test_indices)]
}, index=test_indices)

df['tokens_string'] = df.tokens.apply(
    lambda t: spanish_news_tokenizer.convert_ids_to_tokens(t)
)

df = df[["texto", "tokens", "tokens_string", "categoría", "predicción"]]

df.head(15)

In [ ]:
errors = df[df['categoría'] != df['predicción']]
errors.head(15)

## Comparación con un modelo preentrenado (BERT)

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

model_bert = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=spanish_news_dataset.num_classes
)

El modelo BERT fue cargado desde el checkpoint bert-base-uncased, el cual ha sido previamente entrenado en grandes corpus de texto. Durante la inicialización del modelo para la tarea de clasificación de noticias, las capas finales de clasificación (classifier.weight y classifier.bias) fueron inicializadas aleatoriamente, ya que el modelo original no estaba entrenado para este conjunto específico de categorías. Por esta razón, es necesario realizar un proceso de fine-tuning utilizando el dataset de noticias para adaptar el modelo a la tarea de clasificación.

#Interpretación
El modelo Transformer implementado en este notebook fue entrenado desde cero utilizando únicamente el dataset de noticias. Como se observa en los resultados, el modelo alcanzó una accuracy aproximada del 25% en el conjunto de prueba, lo cual indica que el modelo presenta limitaciones para capturar adecuadamente las relaciones semánticas entre palabras cuando se entrena con un volumen reducido de datos.

En contraste, modelos preentrenados como BERT han sido entrenados previamente en grandes corpus de texto y aprenden representaciones lingüísticas más robustas. Esto permite que, mediante un proceso de fine-tuning, el modelo se adapte a nuevas tareas de clasificación con un mejor desempeño.

In [ ]:
sample = df.iloc[0]

print("Texto:")
print(sample["texto"])

print("\nCategoría real:", sample["categoría"])
print("Predicción del modelo:", sample["predicción"])

In [ ]:
tokens = sample["tokens_string"]
print(tokens)

In [ ]:
import pandas as pd

tokens_df = pd.DataFrame({
    "Token": tokens
})

tokens_df

Tu modelo obtuvo aproximadamente:

Accuracy ≈ 0.25

Eso es 25%.

Si el dataset tiene 4 categorías, sería casi aleatorio.

Si tiene 5-7 categorías, sigue siendo bajo.

Puedes decir algo como:

El desempeño moderado del modelo sugiere que entrenar un Transformer desde cero requiere conjuntos de datos más grandes o mayor número de épocas para alcanzar un rendimiento competitivo.

## Conclusiones
A partir del experimento realizado se pueden establecer las siguientes conclusiones:

1. El modelo Transformer implementado logró entrenarse correctamente, procesando
el texto mediante embeddings con codificación posicional y un bloque de atención.

2. El desempeño obtenido fue aproximadamente del 25% de precisión, lo cual indica que el modelo logra identificar algunos patrones en el texto, aunque aún presenta limitaciones en la clasificación.

3. El análisis de las predicciones muestra que el modelo tiende a favorecer ciertas categorías, especialmente science/technology, lo que sugiere un posible desbalance en el aprendizaje de clases.

4. Las métricas de entrenamiento, validación y prueba son similares, lo cual indica que no existe un sobreajuste significativo, pero también evidencia que el modelo aún no logra capturar completamente la estructura semántica del dataset.

5. Para mejorar el desempeño del modelo podrían considerarse estrategias como:
*  aumentar el número de épocas de entrenamiento
*  utilizar arquitecturas más profundas de Transformer
*  aplicar técnicas de balanceo de clases
*  utilizar modelos preentrenados como BERT o RoBERTa
*  ajustar hiperparámetros como la dimensión del embedding o el número de       
   cabezas de atención.
6. En este experimento se implementó un modelo basado en Transformer para la clasificación de noticias en español. Aunque el modelo logró aprender patrones del dataset, su desempeño fue limitado debido a que fue entrenado desde cero. Esto evidencia la importancia de los modelos preentrenados en tareas modernas de procesamiento de lenguaje natural.

Como extensión del trabajo, se introdujo el modelo BERT, el cual utiliza transfer learning para aprovechar conocimiento lingüístico aprendido previamente. Este tipo de modelos ha demostrado ser altamente efectivo en tareas de clasificación de texto, análisis de sentimiento y comprensión del lenguaje natural.








